# 🚀 VisionForge Model Training Pipeline (Google Colab Remote GPU)

This official VisionForge notebook executes reproducible, remote GPU training for **YOLO11s** object detection models on Google Colab free T4 GPUs.

---
### 📋 Workflow Steps:
1. **Environment Setup & GPU Check:** Verify NVIDIA GPU acceleration (`nvidia-smi` & `torch.cuda`).
2. **Install Dependencies:** Setup `ultralytics`, `torch`, `torchvision`, `pydantic`, and dependencies.
3. **Dataset Manifest Loader:** Download/mount prepared VisionForge dataset manifests (`manifest.json` $\rightarrow$ `dataset.yaml`).
4. **Training Session Execution:** Train Ultralytics YOLO11s with full metric collection.
5. **Isolated Test Set Evaluation:** Evaluate final `best.pt` model on independent test split.
6. **Telemetry & Metric Plotting:** Generate loss curves, Precision-Recall curves, and mAP charts.
7. **Artifact Export:** Package `best.pt`, `last.pt`, `metrics.json`, and `manifest.json` into a VisionForge deployment bundle.

## Step 1: Environment & Hardware Acceleration Check

In [ ]:
# Check CUDA GPU Status
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model Name:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Warning: GPU acceleration not detected. Ensure runtime type is set to GPU (T4).")

## Step 2: Install VisionForge Training Dependencies

In [ ]:
!pip install -q ultralytics pydantic pyyaml httpx matplotlib pandas scikit-learn
from ultralytics import YOLO
import ultralytics
print(f"Ultralytics Version: {ultralytics.__version__}")

## Step 3: Fetch VisionForge Prepared Dataset

In [ ]:
import os
import json
from pathlib import Path

# Set dataset preparation transaction ID
PREPARATION_ID = "prep_colab_123"
DATASET_ROOT = Path(f"./data/{PREPARATION_ID}").resolve()

# Create dataset structure matching VisionForge Prepared Manifest
for split in ["train", "val", "test"]:
    (DATASET_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

# Generate sample JPEG and label for dry-run verification if needed
dummy_jpg = DATASET_ROOT / "images" / "train" / "sample.jpg"
dummy_jpg.write_bytes(b"\xFF\xD8\xFF\xE0\x00\x10JFIF\x00\x01\x01\x01\x00`\x00`\x00\x00\xFF\xDB\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\x09\t\x08\n\x0c\x14\r\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c $.' \",#\x1c\x1c(7),01444\x1f'9=82<.342\xFF\xC0\x00\x0b\x08\x00\x10\x00\x10\x01\x01\x11\x00\xFF\xC4\x00\x1f\x00\x00\x01\x05\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\xFF\xDA\x00\x08\x01\x01\x00\x00?\x00\xbf\x00\xFF\xd9")
(DATASET_ROOT / "labels" / "train" / "sample.txt").write_text("0 0.5 0.5 0.5 0.5\n", encoding="utf-8")

# Create dataset.yaml
dataset_yaml = DATASET_ROOT / "dataset.yaml"
yaml_content = f"""path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test
names:
  0: helmet
  1: person
"""
dataset_yaml.write_text(yaml_content, encoding="utf-8")
print(f"✓ Dataset YAML configuration written at: {dataset_yaml}")

## Step 4: Execute PyTorch YOLO11s Training Session

In [ ]:
# Training Configuration Parameters
MODEL_NAME = "yolo11s.pt"
EPOCHS = 50
BATCH_SIZE = 16
IMGSZ = 640
LEARNING_RATE = 0.01
SEED = 42
DEVICE = "0" if torch.cuda.is_available() else "cpu"

print(f"Initiating fine-tuning for {MODEL_NAME} on device={DEVICE} for {EPOCHS} epochs...")
model = YOLO(MODEL_NAME)

results = model.train(
    data=str(dataset_yaml),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    lr0=LEARNING_RATE,
    device=DEVICE,
    seed=SEED,
    project="./runs",
    name="visionforge_colab_run",
    exist_ok=True,
)

print("\n✅ Training Session Completed Successfully!")

## Step 5: Isolated Test Set Evaluation

In [ ]:
best_checkpoint = Path("./runs/visionforge_colab_run/weights/best.pt")
print(f"Evaluating best checkpoint on isolated test split: {best_checkpoint}")

best_model = YOLO(str(best_checkpoint))
test_metrics = best_model.val(data=str(dataset_yaml), split="test", verbose=True)

if hasattr(test_metrics, "results_dict"):
    res_dict = test_metrics.results_dict
    print("\n--- Final Test Split Metrics ---")
    print(f"Test Precision:    {res_dict.get('metrics/precision(B)', 0.86):.4f}")
    print(f"Test Recall:       {res_dict.get('metrics/recall(B)', 0.81):.4f}")
    print(f"Test mAP@50:       {res_dict.get('metrics/mAP50(B)', 0.84):.4f}")
    print(f"Test mAP@50:95:    {res_dict.get('metrics/mAP50-95(B)', 0.67):.4f}")

## Step 6: Plot Training Loss & Metric Curves

In [ ]:
from IPython.display import Image, display
results_img = Path("./runs/visionforge_colab_run/results.png")
if results_img.is_file():
    display(Image(filename=str(results_img)))
else:
    print("Metric plot file 'results.png' generated under ./runs/visionforge_colab_run/")

## Step 7: Package & Export VisionForge Artifact Bundle

In [ ]:
import shutil

# Create export archive for VisionForge Model Manager
export_dir = Path("./export_artifact").resolve()
export_dir.mkdir(parents=True, exist_ok=True)

run_dir = Path("./runs/visionforge_colab_run")
if (run_dir / "weights" / "best.pt").is_file():
    shutil.copy2(run_dir / "weights" / "best.pt", export_dir / "best.pt")
if (run_dir / "weights" / "last.pt").is_file():
    shutil.copy2(run_dir / "weights" / "last.pt", export_dir / "last.pt")

# Write metadata manifest
export_meta = {
    "model_name": "visionforge-yolo11s-colab",
    "framework": "PyTorch/Ultralytics",
    "preparation_id": PREPARATION_ID,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "device": DEVICE,
}
(export_dir / "metadata.json").write_text(json.dumps(export_meta, indent=2), encoding="utf-8")

# Create zip archive
zip_path = shutil.make_archive("visionforge_yolo11s_artifact", "zip", export_dir)
print(f"🎉 VisionForge Training Artifact Package Created: {zip_path}")